# WLCF Example: 3PCF Multipoles

This notebook runs a compact weak-lensing 3PCF workflow with the Python wrapper. It keeps the parameter surface small, uses the bundled test power spectrum, and produces the same core outputs as the longer exploratory notebook: a halo-model map, one-dimensional slices, and a comparison across tree-level model branches.

All heatmaps use the original plotting bounds, layout, color scale, and angular window so panels can be compared directly.


## Setup

Run this notebook from the repository root or from `tests/`. The setup cell finds the project, switches to `tests/`, imports the wrapper, and defines small plotting helpers.


In [ ]:
from pathlib import Path
import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "Makefile").exists() and (path / "source").exists() and (path / "tests").exists():
            return path
    raise RuntimeError("Could not find the wlcf repository root. Open this notebook from the repository or tests folder.")


REPO_ROOT = find_repo_root()
TEST_DIR = REPO_ROOT / "tests"
os.chdir(TEST_DIR)

try:
    from wlcfpy import wlcf
except ImportError as exc:
    raise ImportError(
        "Could not import wlcfpy. Build it from the repository root with "
        "PYTHON=python3 make all, then restart this notebook kernel."
    ) from exc

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

OUTPUT_DIR = Path("Bell_outputs")
ARC_MIN_PER_RAD = 180.0 / np.pi * 60.0

print(f"Repository: {REPO_ROOT}")
print(f"Working directory: {Path.cwd()}")


## Parameters

The example keeps only the knobs that matter for a first run: input file, output names, model branch, resolution, and thread count. The resolution below is intentionally modest so the notebook stays interactive.


In [ ]:
base_params = {
    # Input and output
    "fnamePS": "./input/linear_pk_Takahashi_z0.txt",
    "rootDir": "Output",
    "path_Bells": str(OUTPUT_DIR),

    # Runtime
    "numberThreads": 8,
    "verbose": 1,
    "verbose_log": 0,

    # Compact notebook resolution
    "mMax": 4,
    "Nell": 64,
    "chiQuadSteps": 120,
    "GLpoints": 32,
    "writevectors": 0,
}

moments = range(5)
slice_moments = [0, 1, 2]
theta2_targets = [13, 72, 169]
PLOT_THETA_LIMITS = (10, 200)
COLOR_SCALE_MIN = 1e-11
COLOR_SCALE_MAX = 1e-8


def run_wlcf(prefix, tree_level):
    params = dict(base_params, prefix=prefix, tree_level=tree_level)
    runner = wlcf()
    runner.set(params)
    cpu_time = runner.Run()
    print(f"tree_level={tree_level:<2} prefix={prefix:<16} cpu/thread={cpu_time:.3f}s")
    return params


def load_zetas(prefix, selected_moments=moments):
    theta = np.loadtxt(OUTPUT_DIR / f"{prefix}theta_array.txt") * ARC_MIN_PER_RAD
    zetas = {
        m: np.abs(np.loadtxt(OUTPUT_DIR / f"{prefix}zetam{m}.txt"))
        for m in selected_moments
    }
    return theta, zetas


def shared_log_norm(collection, vmin=COLOR_SCALE_MIN, vmax=COLOR_SCALE_MAX):
    values = np.concatenate([arr.ravel() for item in collection for arr in item.values()])
    values = values[np.isfinite(values) & (values > 0)]
    if values.size == 0:
        raise ValueError("No positive finite values available for LogNorm.")
    return LogNorm(vmin=vmin, vmax=vmax)


## Run The Halo-Model Case

This mirrors the most common quick-look run: `tree_level=4`, using a dedicated prefix so outputs are easy to identify.


In [ ]:
halo_prefix = "example_halo_"
halo_params = run_wlcf(prefix=halo_prefix, tree_level=4)

theta, halo_zetas = load_zetas(halo_prefix)
print(f"theta range: {theta.min():.2f} to {theta.max():.2f} arcmin")
print(f"grid shape: {next(iter(halo_zetas.values())).shape}")


## Halo-Model Multipole Maps

The first three multipoles use one shared color scale.


In [ ]:
halo_norm = shared_log_norm([halo_zetas])
Theta2, Theta1 = np.meshgrid(theta, theta)

fig = plt.figure(figsize=(19, 4))
gs = fig.add_gridspec(
    nrows=1,
    ncols=len(moments) + 1,
    width_ratios=[1] * len(moments) + [0.06],
    wspace=0.25,
)

for col, m in enumerate(moments):
    ax = fig.add_subplot(gs[0, col])
    mesh = ax.pcolormesh(
        Theta2,
        Theta1,
        halo_zetas[m],
        shading="auto",
        cmap="RdYlBu_r",
        norm=halo_norm,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*PLOT_THETA_LIMITS)
    ax.set_ylim(*PLOT_THETA_LIMITS)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(fr"$m={m}$", fontsize=14)
    ax.set_xlabel(r"$\theta_2$ [arcmin]", fontsize=13)

    if col == 0:
        ax.set_ylabel(r"$\theta_1$ [arcmin]", fontsize=13)

cax = fig.add_subplot(gs[0, -1])
cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label(r"$|\zeta_m|$", fontsize=13)
cbar.ax.tick_params(labelsize=11)

fig.suptitle("Halo Model: 3PCF Multipoles", fontsize=18, y=1.02)
plt.show()


## Halo-Model Slices

Each panel fixes `theta_2` near a target angle and varies `theta_1`.


In [ ]:
fig, axes = plt.subplots(
    len(theta2_targets),
    len(slice_moments),
    figsize=(9, 9),
    sharex=True,
)

for row, theta2_target in enumerate(theta2_targets):
    idx = int(np.argmin(np.abs(theta - theta2_target)))
    theta2_actual = theta[idx]

    for col, m in enumerate(slice_moments):
        ax = axes[row, col]
        ax.plot(theta, halo_zetas[m][:, idx], color="black", marker="o", markersize=3, lw=1)

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(*PLOT_THETA_LIMITS)
        ax.grid(alpha=0.2)

        if row == 0:
            ax.set_title(fr"$m={m}$")

        if col == 0:
            ax.set_ylabel(fr"$\zeta_m(\theta_1,\theta_2={theta2_actual:.0f}')$")

        if row == len(theta2_targets) - 1:
            ax.set_xlabel(r"$\theta_1$ [arcmin]")

plt.tight_layout()
plt.show()


## Compare Tree-Level Branches

The same compact configuration is rerun for the four model branches. Each model gets a unique prefix, and the comparison plot uses one color scale across every row and column.


In [ ]:
models = {
    "SPT": {"tree_level": 1, "prefix": "example_spt_"},
    "Tree": {"tree_level": 2, "prefix": "example_tree_"},
    "EFT": {"tree_level": 3, "prefix": "example_eft_"},
    "Halo Model": {"tree_level": 4, "prefix": "example_halo_"},
}

model_zetas = {}
for name, cfg in models.items():
    run_wlcf(prefix=cfg["prefix"], tree_level=cfg["tree_level"])
    model_theta, model_zetas[name] = load_zetas(cfg["prefix"])

comparison_norm = shared_log_norm(model_zetas.values())


## Model Comparison

Rows are model branches and columns are multipoles. The shared colorbar makes the model-to-model differences visually meaningful.


In [ ]:
with plt.rc_context({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14}):
    Theta2, Theta1 = np.meshgrid(model_theta, model_theta)
    nrows = len(models)
    ncols = len(moments)

    fig = plt.figure(figsize=(20, 13))
    gs = fig.add_gridspec(
        nrows=nrows,
        ncols=ncols + 1,
        width_ratios=[1] * ncols + [0.05],
        wspace=0.2,
        hspace=0.25,
    )

    for row, (model_name, zetas) in enumerate(model_zetas.items()):
        for col, m in enumerate(moments):
            ax = fig.add_subplot(gs[row, col])
            mesh = ax.pcolormesh(
                Theta2,
                Theta1,
                zetas[m],
                shading="auto",
                cmap="RdYlBu_r",
                norm=comparison_norm,
            )
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlim(*PLOT_THETA_LIMITS)
            ax.set_ylim(*PLOT_THETA_LIMITS)

            if row == 0:
                ax.set_title(fr"$m={m}$")
            if col == 0:
                ax.set_ylabel(f"{model_name}\n" + r"$\theta_1$ [arcmin]")
            if row == nrows - 1:
                ax.set_xlabel(r"$\theta_2$ [arcmin]")

    cax = fig.add_subplot(gs[:, -1])
    cbar = fig.colorbar(mesh, cax=cax)
    cbar.set_label(r"$|\zeta_m|$", fontsize=16)
    cbar.ax.tick_params(labelsize=12)
plt.show()


## Next Steps

For a higher-accuracy run, increase `Nell`, `chiQuadSteps`, and `GLpoints`. For a production analysis, replace `fnamePS` with a power spectrum generated for the cosmology and redshift of interest.
